### Import libraries

In [1]:
import re
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

RANDOM_STATE = 42

print("Libraries imported successfully!")

Libraries imported successfully!


### Set project paths and configuration

In [ ]:
# Find the project root directory
ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

# Dataset paths
RAW_DATA_PATH = ROOT / "data" / "raw" / "amazon_mobile_reviews.csv"

PROCESSED_DIR = ROOT / "data" / "processed"
INTERIM_DIR = ROOT / "data" / "interim"

# Create output folders if they do not exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# Change these if your dataset uses different column names
TEXT_COLUMN = "text"
LABEL_COLUMN = "sentiment"

print("Project root:", ROOT)
print("Raw dataset:", RAW_DATA_PATH)
print("Processed folder:", PROCESSED_DIR)
print("Raw dataset exists:", RAW_DATA_PATH.exists())

### Load the raw dataset

In [ ]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {RAW_DATA_PATH}\n"
        "Please check the dataset filename and folder."
    )

df = pd.read_csv(RAW_DATA_PATH)

print("Dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.head())

### Understand the dataset

In [2]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)

NameError: name 'df' is not defined

### Check missing values

In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (
        df.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    by="Missing_Count",
    ascending=False
)

display(missing_summary)

### Validate required columns

In [ ]:
required_columns = [TEXT_COLUMN, LABEL_COLUMN]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Required columns not found: {missing_columns}\n"
        f"Available columns: {df.columns.tolist()}\n"
        "Update TEXT_COLUMN and LABEL_COLUMN in Cell 2."
    )

print("Required columns are available.")

### Standardize column names

In [ ]:
df = df.rename(columns={
    TEXT_COLUMN: "text",
    LABEL_COLUMN: "sentiment"
})

# Keep the original text and sentiment columns
df["text"] = df["text"].astype("string")
df["sentiment"] = df["sentiment"].astype("string").str.strip()

print("Standardized columns:")
print(df.columns.tolist())

display(df[["text", "sentiment"]].head())

### Check sentiment labels

In [ ]:
print("Unique sentiment labels:")
print(df["sentiment"].value_counts(dropna=False))

print("\nNumber of unique labels:")
print(df["sentiment"].nunique())

### Clean sentiment labels

In [ ]:
df["sentiment"] = df["sentiment"].str.strip().str.lower()

# Standardize common label variations
label_mapping = {
    "positive": "positive",
    "pos": "positive",
    "negative": "negative",
    "neg": "negative",
    "neutral": "neutral",
    "neu": "neutral"
}

df["sentiment"] = df["sentiment"].replace(label_mapping)

VALID_LABELS = ["positive", "negative", "neutral"]

invalid_labels = ~df["sentiment"].isin(VALID_LABELS)

print("Invalid or missing labels:", invalid_labels.sum())

# Remove rows with missing or unsupported labels
df = df.loc[~invalid_labels].copy()

print("\nSentiment labels after cleaning:")
print(df["sentiment"].value_counts())

### Clean text data

In [ ]:
def clean_text(text):
    """
    Clean text while keeping useful words and punctuation.
    """

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # Remove user mentions
    text = re.sub(r"@\w+", " ", text)

    # Keep words from hashtags
    text = re.sub(r"#(\w+)", r"\1", text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove email addresses
    text = re.sub(
        r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b",
        " ",
        text
    )

    # Remove unwanted characters but preserve common punctuation
    text = re.sub(
        r"[^\w\s.,!?'-]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["clean_text"] = df["text"].apply(clean_text)

print("Text cleaning completed!")

display(df[["text", "clean_text", "sentiment"]].head(10))

### Check empty text after cleaning

In [ ]:
empty_text_mask = (
    df["clean_text"].isna()
    | df["clean_text"].str.strip().eq("")
)

print("Empty reviews:", empty_text_mask.sum())

# Remove reviews that contain no usable text
df = df.loc[~empty_text_mask].copy()

df.reset_index(drop=True, inplace=True)

print("Dataset shape after removing empty reviews:", df.shape)

### Check duplicate reviews

In [ ]:
# Exact duplicate rows
print("Exact duplicate rows:", df.duplicated().sum())

# Same cleaned review with the same sentiment
print(
    "Duplicate text-label pairs:",
    df.duplicated(subset=["clean_text", "sentiment"]).sum()
)

# Same text with conflicting sentiment labels
label_counts_per_text = df.groupby("clean_text")["sentiment"].nunique()

conflicting_texts = label_counts_per_text[
    label_counts_per_text > 1
]

print(
    "Reviews with conflicting sentiment labels:",
    len(conflicting_texts)
)

### Remove duplicate records

In [ ]:
before = len(df)

# Remove identical text-label pairs
df = df.drop_duplicates(
    subset=["clean_text", "sentiment"]
).reset_index(drop=True)

print("Duplicate records removed:", before - len(df))
print("Dataset shape:", df.shape)

### Analyze sentiment distribution

In [ ]:
sentiment_counts = df["sentiment"].value_counts()

display(
    sentiment_counts.rename("Count").to_frame()
)

sentiment_percentages = (
    df["sentiment"].value_counts(normalize=True) * 100
).round(2)

print("\nSentiment percentages:")
display(sentiment_percentages.rename("Percentage").to_frame())

### Visualize sentiment distribution

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="sentiment",
    order=["positive", "neutral", "negative"]
)

plt.title("Sentiment Class Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()

### Analyze review lengths

In [ ]:
df["word_count"] = df["clean_text"].str.split().str.len()
df["character_count"] = df["clean_text"].str.len()

display(
    df[["word_count", "character_count"]].describe().T
)

plt.figure(figsize=(9, 5))

sns.histplot(
    data=df,
    x="word_count",
    bins=40,
    kde=True
)

plt.title("Distribution of Review Lengths")
plt.xlabel("Number of Words")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()

### Check the cleaned dataset

In [ ]:
print("Final cleaned dataset shape:", df.shape)

print("\nMissing values:")
print(df[["text", "clean_text", "sentiment"]].isna().sum())

print("\nEmpty cleaned reviews:")
print(df["clean_text"].str.strip().eq("").sum())

print("\nDuplicate text-label pairs:")
print(df.duplicated(
    subset=["clean_text", "sentiment"]
).sum())

print("\nFinal sentiment distribution:")
print(df["sentiment"].value_counts())

### Prepare data for splitting

In [ ]:
# Keep the columns required for model training
model_df = df[
    ["text", "clean_text", "sentiment"]
].copy()

model_df = model_df.reset_index(drop=True)

print("Data ready for splitting:", model_df.shape)

display(model_df.head())

### Split into training, validation, and testing sets

In [ ]:
# First split: 80% training and 20% temporary data
train_df, temp_df = train_test_split(
    model_df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=model_df["sentiment"]
)

# Second split: divide the remaining 20% equally
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["sentiment"]
)

print("Training dataset:", train_df.shape)
print("Validation dataset:", val_df.shape)
print("Testing dataset:", test_df.shape)

### Verify split percentages

In [ ]:
total_records = len(model_df)

split_summary = pd.DataFrame({
    "Dataset": ["Training", "Validation", "Testing"],
    "Records": [
        len(train_df),
        len(val_df),
        len(test_df)
    ]
})

split_summary["Percentage"] = (
    split_summary["Records"] / total_records * 100
).round(2)

display(split_summary)

assert (
    len(train_df) + len(val_df) + len(test_df)
    == total_records
), "Dataset records do not match!"

### Compare sentiment distribution across splits

In [ ]:
distribution = pd.concat(
    [
        train_df["sentiment"].value_counts(normalize=True).rename("Train"),
        val_df["sentiment"].value_counts(normalize=True).rename("Validation"),
        test_df["sentiment"].value_counts(normalize=True).rename("Test")
    ],
    axis=1
).fillna(0) * 100

distribution = distribution.round(2)

display(distribution)

distribution.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Sentiment Distribution Across Dataset Splits")
plt.xlabel("Sentiment")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Check for data leakage

In [ ]:
train_texts = set(train_df["clean_text"])
val_texts = set(val_df["clean_text"])
test_texts = set(test_df["clean_text"])

print("Train-validation overlap:", len(train_texts & val_texts))
print("Train-test overlap:", len(train_texts & test_texts))
print("Validation-test overlap:", len(val_texts & test_texts))

assert train_texts.isdisjoint(val_texts)
assert train_texts.isdisjoint(test_texts)
assert val_texts.isdisjoint(test_texts)

print("\nNo duplicate cleaned reviews across splits.")

### Save the cleaned dataset

In [ ]:
cleaned_path = PROCESSED_DIR / "cleaned_reviews.csv"

df.to_csv(
    cleaned_path,
    index=False,
    encoding="utf-8"
)

print("Cleaned dataset saved to:", cleaned_path)

### Save training, validation, and testing datasets

In [ ]:
train_path = PROCESSED_DIR / "train.csv"
val_path = PROCESSED_DIR / "validation.csv"
test_path = PROCESSED_DIR / "test.csv"

train_df.to_csv(train_path, index=False, encoding="utf-8")
val_df.to_csv(val_path, index=False, encoding="utf-8")
test_df.to_csv(test_path, index=False, encoding="utf-8")

print("All datasets saved successfully!")

print("Training:", train_path)
print("Validation:", val_path)
print("Testing:", test_path)

Final verification

In [ ]:
# Reload saved datasets to verify the files
saved_train = pd.read_csv(train_path)
saved_val = pd.read_csv(val_path)
saved_test = pd.read_csv(test_path)

print("Final verification")
print("-" * 40)

print("Training records:", len(saved_train))
print("Validation records:", len(saved_val))
print("Testing records:", len(saved_test))

print(
    "Total saved records:",
    len(saved_train) + len(saved_val) + len(saved_test)
)

print("\nSaved files:")
for path in [cleaned_path, train_path, val_path, test_path]:
    print(f"{path.name}: {'Exists' if path.exists() else 'Missing'}")

print("\nPreprocessing notebook completed!")